In [50]:
import torch
from PIL import Image
import pandas as pd
from transformers import AutoProcessor, Blip2Processor, Blip2ForImageTextRetrieval,  Blip2ForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm
from transformers import MarianMTModel, MarianTokenizer
import os
import translate_dn_en
import numpy as np

from translate_dn_en import translate_danish_to_english

In [51]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [52]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-flan-t5-xl")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [54]:
df = pd.read_csv("metadata.csv")

df = df.dropna(subset=["Image File", "Category"]) # drop rows with missing values
# use the first 100, can be changed for other subsets
df = df[:10]
true_labels = list(df["Category"])
categories = list(df["Category"].unique())

image_paths = [
    os.path.join("images2", fname) # change images2 to name of folder with images
    for fname in df["Image File"]
]

In [55]:
category_to_en = {
    cat : translate_danish_to_english(cat)
    for cat in categories
}


In [56]:
category_to_en

{'Fritid & Have': 'Recreation & Garden',
 'Lamper': 'Lamps',
 'Elektronik': 'Electronics',
 'Boligting': 'Housing',
 'Musik & Bøger': 'Music & Books'}

In [57]:
df["Category_en"] = df["Category"].map(category_to_en)
df["Category_en"]

0    Recreation & Garden
1                  Lamps
2            Electronics
3    Recreation & Garden
4                Housing
5                Housing
6    Recreation & Garden
7          Music & Books
8                  Lamps
9          Music & Books
Name: Category_en, dtype: object

In [58]:
prompt = f"Given an image, output ONLY ONE Category from this list: {', '.join(df["Category_en"])}. "
prompt

'Given an image, output ONLY ONE Category from this list: Recreation & Garden, Lamps, Electronics, Recreation & Garden, Housing, Housing, Recreation & Garden, Music & Books, Lamps, Music & Books. '

In [59]:
def classify_image(image_path: str, prompt: str):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=prompt, return_tensors="pt", padding=True)
    
    generated_ids = model.generate(  
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    prediction = processor.decode(
        generated_ids[0], skip_special_tokens=True
    )
    print(prediction)
    return prediction

In [63]:
# TODO : make it work with new english prompts
correct = 0

for img_path, gt in zip(image_paths, true_labels):
    pred = classify_image(img_path, prompt)
    if pred == gt:
        correct += 1

accuracy = correct / len(image_paths)
print("Accuracy:", accuracy)

Recreation & Garden
Lamps
Electronics
Recreation & Garden
Recreation & Garden
Recreation & Garden
Recreation & Garden
Recreation & Garden
Lamps
Recreation & Garden
Accuracy: 0.0
